In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_36687/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,John Collins,Over,14.0,-137,2025-11-21,2025-11-20T23:41:20Z
1,PrizePicks,player_points,John Collins,Under,14.0,-137,2025-11-21,2025-11-20T23:41:20Z
2,PrizePicks,player_points,James Harden,Over,27.5,-137,2025-11-21,2025-11-20T23:41:20Z
3,PrizePicks,player_points,James Harden,Under,27.5,-137,2025-11-21,2025-11-20T23:41:20Z
4,PrizePicks,player_points,Franz Wagner,Over,23.5,-137,2025-11-21,2025-11-20T23:41:20Z


### Update projected starting lineups

In [4]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 8 teams with confirmed lineups


### Top EVs for single bets

In [13]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

singleBets = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)



singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION', 'SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets...
Pre-computing predictions for 58 unique players...
Error getting prediction for Paul George: float division by zero


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
466,Justin Edwards,DraftKings,6.5,9.96,Over,100,0,5.55,0.555,Low
496,Jalen Johnson,BetMGM,22.5,25.84,Over,100,0,3.84,0.384,High
429,Santi Aldama,DraftKings,18.5,14.52,Under,-118,0,3.67,0.432,High
201,Julian Champagnie,BetRivers,10.5,7.99,Under,102,0,3.37,0.330,High
71,Cedric Coward,FanDuel,17.5,13.52,Under,-125,0,3.26,0.408,High


## Top EVs for 2 leg bets

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['COMMENCE_TIME'] == '2025-11-21')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 38 players...
Error getting prediction for Paul George: float division by zero
Processing 32 players...
Generated 440 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
345,Nickeil Alexander-Walker,Santi Aldama,16.5,17.5,20.73,14.52,0.728,0.685,over,under,0,4.64,0.232,High,High
251,Jalen Johnson,Tyrese Maxey,22.5,29.5,25.84,26.30,0.692,0.674,over,under,0,3.71,0.186,High,High
147,Tristan da Silva,Luke Kornet,11.5,9.5,14.14,7.26,0.667,0.685,over,under,0,3.44,0.172,High,Low
168,Kobe Sanders,Julian Champagnie,9.5,10.5,7.73,7.99,0.635,0.662,under,under,0,2.35,0.117,Med,High
272,Keldon Johnson,Bobby Portis,14.5,14.5,11.94,12.32,0.660,0.635,under,under,0,2.32,0.116,High,High
230,Devin Vassell,Malik Monk,16.5,13.5,14.26,11.31,0.630,0.628,under,under,0,1.62,0.081,High,High
313,Luke Kennard,Russell Westbrook,5.5,13.5,7.04,15.66,0.622,0.625,over,over,0,1.44,0.072,Low,High
90,Bogdan Bogdanović,Jeremy Sochan,11.5,8.5,9.59,7.00,0.619,0.612,under,under,0,1.14,0.057,High,Med
4,James Harden,De'Aaron Fox,27.5,24.5,25.73,23.09,0.603,0.583,under,under,0,0.34,0.017,High,High
331,Onyeka Okongwu,Kyle Kuzma,13.5,15.5,14.65,16.92,0.568,0.581,over,over,0,-0.30,0.000,High,High


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')& (dfsData['COMMENCE_TIME'] == '2025-11-21')]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 58 players...
Error getting prediction for Paul George: float division by zero
Processing 52 players...
Generated 1180 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV$,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
1014,Santi Aldama,Justin Edwards,18.5,6.5,14.52,9.96,under,over,0.740,0.778,0.5638,0.162,0.200,0.241,6.91,0.346,0,6.19,4.53,High,Low,"(2.4, 26.7)","(1.1, 18.8)",0.05,0,69.1
651,Nickeil Alexander-Walker,Cedric Coward,16.5,16.5,20.73,13.52,over,under,0.728,0.682,0.4866,0.149,0.104,0.162,4.60,0.230,0,7.00,6.29,High,High,"(7.0, 34.4)","(1.2, 25.8)",0.05,0,46.0
630,Jalen Johnson,Tyrese Maxey,22.5,29.5,25.84,26.30,over,under,0.692,0.674,0.4572,0.114,0.096,0.132,3.71,0.186,0,6.66,7.09,High,High,"(12.8, 38.9)","(12.4, 40.2)",0.05,0,37.1
318,Tristan da Silva,Luke Kornet,11.5,9.5,14.14,7.26,over,under,0.667,0.685,0.4479,0.089,0.107,0.123,3.44,0.172,0,6.11,4.66,High,Low,"(2.2, 26.1)","(0.0, 16.4)",0.05,0,34.4
858,Zaccharie Risacher,Kyle Kuzma,11.5,14.5,14.11,16.92,over,over,0.663,0.636,0.4134,0.085,0.058,0.088,2.40,0.120,0,6.21,6.93,High,High,"(1.9, 26.3)","(3.3, 30.5)",0.05,0,24.0
357,Kobe Sanders,Julian Champagnie,9.5,10.5,7.73,7.99,under,under,0.635,0.662,0.4116,0.057,0.084,0.086,2.35,0.117,0,5.14,6.03,Med,High,"(0.0, 17.8)","(0.0, 19.8)",0.05,0,23.5
748,Keldon Johnson,Bobby Portis,14.5,14.5,11.94,12.32,under,under,0.660,0.635,0.4107,0.082,0.056,0.085,2.32,0.116,0,6.18,6.34,High,High,"(0.0, 24.1)","(0.0, 24.7)",0.05,0,23.2
719,Kristaps Porziņģis,Myles Turner,16.5,16.5,18.56,14.35,over,under,0.631,0.628,0.3880,0.053,0.050,0.062,1.64,0.082,0,6.16,6.60,High,High,"(6.5, 30.6)","(1.4, 27.3)",0.05,0,16.4
680,Devin Vassell,Malik Monk,16.5,13.5,14.26,11.31,under,under,0.630,0.628,0.3874,0.052,0.050,0.061,1.62,0.081,0,6.77,6.73,High,High,"(1.0, 27.5)","(0.0, 24.5)",0.05,0,16.2
525,Goga Bitadze,Russell Westbrook,4.5,13.5,6.00,15.66,over,over,0.626,0.625,0.3837,0.048,0.047,0.057,1.51,0.076,0,4.66,6.77,Low,High,"(0.0, 15.1)","(2.4, 28.9)",0.05,0,15.1


## 3 leg parlay

### Underdog picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['COMMENCE_TIME'] == '2025-11-21')]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 38 players...
Error getting prediction for Paul George: float division by zero
Processing 32 players...
Generated 4666 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
3350,Jalen Johnson,Nickeil Alexander-Walker,Santi Aldama,22.5,16.5,17.5,25.84,20.73,14.52,0.692,0.728,0.685,over,over,under,0,8.61,0.172,High,High,High
2226,Tristan da Silva,Luke Kornet,Tyrese Maxey,11.5,9.5,29.5,14.14,7.26,26.30,0.667,0.685,0.674,over,under,under,0,6.64,0.133,High,Low,High
2416,Kobe Sanders,Keldon Johnson,Julian Champagnie,9.5,14.5,10.5,7.73,11.94,7.99,0.635,0.660,0.662,under,under,under,0,4.98,0.100,Med,High,High
3227,Devin Vassell,Malik Monk,Bobby Portis,16.5,13.5,14.5,14.26,11.31,12.32,0.630,0.628,0.635,under,under,under,0,3.54,0.071,High,High,High
1502,Bogdan Bogdanović,Luke Kennard,Russell Westbrook,11.5,5.5,13.5,9.59,7.04,15.66,0.619,0.622,0.625,under,over,over,0,3.02,0.060,High,Low,High


### Prizepicks picks

In [10]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')& (dfsData['COMMENCE_TIME'] == '2025-11-21')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 58 players...
Error getting prediction for Paul George: float division by zero
Processing 52 players...
Generated 20896 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
14745,Nickeil Alexander-Walker,Santi Aldama,Justin Edwards,16.5,18.5,6.5,20.73,14.52,9.96,0.728,0.740,0.778,over,under,over,0,12.60,0.252,High,High,Low
14245,Jalen Johnson,Cedric Coward,Tyrese Maxey,22.5,16.5,29.5,25.84,13.52,26.30,0.692,0.682,0.674,over,under,under,0,7.19,0.144,High,High,High
8134,Tristan da Silva,Julian Champagnie,Luke Kornet,11.5,10.5,9.5,14.14,7.99,7.26,0.667,0.662,0.685,over,under,under,0,6.33,0.127,High,High,Low
9022,Kobe Sanders,Zaccharie Risacher,Kyle Kuzma,9.5,11.5,14.5,7.73,14.11,16.92,0.635,0.663,0.636,under,over,over,0,4.46,0.089,Med,High,High
14995,Devin Vassell,Keldon Johnson,Bobby Portis,16.5,14.5,14.5,14.26,11.94,12.32,0.630,0.660,0.635,under,under,under,0,4.25,0.085,High,High,High
